# EDA phân bố nhãn (B7)

> Mục đích: Xem nhãn B7 của mẫu khám phá nhãn bằng bảng và biểu đồ: từng chiều, thái độ, nhóm, chỗ A chưa chắc, đối chiếu với mã tự do; cập nhật lần cuối: 2026-11-25.

- Luật: `docs/3-gan-nhan/dac-ta-kham-pha-nhan.md` mục 7(c), 8. Code: `src/labeling/b7_labels.py`, `src/labeling/distribution.py`.
- Nhãn do AI (A) gán. **B chưa kiểm xong `phieu_kiem_b7.csv` thì mọi biểu đồ chỉ là số đếm.** Kiểm xong, chạy lại thì có phần trăm và khoảng tin cậy 95% (bootstrap theo video).
- Notebook không in chữ bình luận.


In [ ]:
from helpers import *
import matplotlib.pyplot as plt
import pandas as pd
from src.labeling import distribution as D
from src.labeling.b7_labels import CHIEU, THAI_DO
from src.labeling.label_set import NGOAI_KHIA_CANH
from src.labeling.sample import doc_csv

dong, kiem, goc = D.doc()
tk = D.thong_ke(dong)
MO = kiem["xong"]                      # đã kiểm xong mới được hiện phần trăm
print(f"n = {tk['n']} bình luận, {tk['so_video']} video; đã kiểm {kiem['da_kiem']}/{kiem['can_kiem']}" + ("" if MO else "  ->  CHỈ SỐ ĐẾM"))

## 1. Hàm vẽ

In [ ]:
XANH, DO, CHU, CHU_NHAT, NEN = "#2a78d6", "#e34948", "#0b0b0b", "#52514e", "#fcfcfb"

def _khung(ax, ten, truc):
    ax.set_facecolor(NEN)
    ax.figure.set_facecolor(NEN)
    ax.set_title(ten, loc="left", color=CHU, fontsize=11)
    ax.set_xlabel(truc, color=CHU_NHAT)
    for c in ("top", "right", "left"):
        ax.spines[c].set_visible(False)
    ax.tick_params(axis="y", length=0, colors=CHU)
    ax.tick_params(axis="x", colors=CHU_NHAT)
    ax.grid(axis="x", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)

def ve_nhan(b, ten, mau_so):
    """Thanh ngang một màu: số đếm; đã kiểm thì là % kèm khoảng bootstrap theo video."""
    muc = sorted(b.items(), key=lambda u: u[1]["x"])
    y = range(len(muc))
    fig, ax = plt.subplots(figsize=(8, 0.34 * len(muc) + 1.2))
    gt = [100 * k["p"] if MO else k["x"] for _, k in muc]
    ax.barh(y, gt, height=0.6, color=XANH)
    if MO:
        ax.hlines(y, [100 * k["bl"] for _, k in muc], [100 * k["bu"] for _, k in muc], color=CHU, linewidth=1.5)
    for i, (v, (_, k)) in enumerate(zip(gt, muc)):
        x = max(v, 100 * k["bu"]) if MO else v
        ax.annotate(f"{v:.1f}%" if MO else str(v), (x, i), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9, color=CHU_NHAT)
    ax.set_yticks(list(y), [x for x, _ in muc])
    _khung(ax, f"{ten} (n = {mau_so})", "% bình luận, vạch đen: KTC 95%" if MO else "số bình luận")
    plt.tight_layout(); plt.show()

## 2. Sáu chiều

In [ ]:
ve_nhan(tk["lien_quan"], CHIEU[1][1], tk["n"])
ve_nhan(tk["noi_ve"], CHIEU[2][1], f"{tk['n_dt']} dien_thoai")
ve_nhan(tk["muc_dich"], CHIEU[3][1], tk["n"])
ve_nhan(tk["da_dung"], CHIEU[5][1], tk["n"])
ve_nhan(tk["may_nhac"], CHIEU[6][1], tk["n"])

## 3. Thái độ theo nhãn chiều 2

Chê bên trái, khen bên phải, tính theo cặp (bình luận, nhãn chiều 2). Không vẽ `so_sanh`, `khong_danh_gia`; xem bảng dưới. Nhãn có khen + chê dưới 30 thì **chưa kết luận** được khen hay chê nhiều hơn.

In [ ]:
kc = [k for k, _ in tk["so_cap"].most_common()][::-1]
khen = [tk["cap"].get((k, "tich_cuc"), 0) for k in kc]
che = [tk["cap"].get((k, "tieu_cuc"), 0) for k in kc]
fig, ax = plt.subplots(figsize=(8, 0.36 * len(kc) + 1.4))
ax.barh(range(len(kc)), khen, height=0.6, color=XANH, label="khen (tich_cuc)")
ax.barh(range(len(kc)), [-c for c in che], height=0.6, color=DO, label="chê (tieu_cuc)")
for i, (a, b) in enumerate(zip(khen, che)):
    if a: ax.annotate(a, (a, i), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9, color=CHU_NHAT)
    if b: ax.annotate(b, (-b, i), xytext=(-4, 0), textcoords="offset points", va="center", ha="right", fontsize=9, color=CHU_NHAT)
m = max(khen + che) + 2
ax.set_xlim(-m, m)
ax.axvline(0, color=CHU_NHAT, linewidth=0.8)
ax.set_xticks(ax.get_xticks(), [f"{abs(int(t))}" for t in ax.get_xticks()])
ax.set_xlim(-m, m)
ax.set_yticks(range(len(kc)), kc)
_khung(ax, f"Khen và chê theo nhãn chiều 2 ({tk['n']} bình luận)", "số cặp")
ax.legend(frameon=False, loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()

pd.DataFrame([{"nhãn": k, "loại": "ngoài khía cạnh" if k in NGOAI_KHIA_CANH else "khía cạnh", "số cặp": tk["so_cap"][k],
               **{t: tk["cap"].get((k, t), 0) for t in THAI_DO}} for k in kc[::-1]])

## 4. Mục đích theo chiều 1 (số bình luận)

In [ ]:
bang = pd.DataFrame({l: {m: tk["lq_md"].get((l, m), 0) for m in tk["muc_dich"]} for l in tk["lien_quan"]})
bang.loc[sorted(bang.index, key=lambda m: -tk["muc_dich"][m]["x"])]

## 5. Theo nhóm

Số bình luận của nhóm có nhãn đó. Nhóm dưới 30 bình luận không tính phần trăm.

In [ ]:
COT = [("lq", x) for x in ("dien_thoai", "ngu_canh", "nguoi_binh_luan")] + [("md", x) for x in ("danh_gia", "hoi", "tranh_luan", "cong_kich", "minigame")] \
    + [("td", "tich_cuc"), ("td", "tieu_cuc")]
for ten, gs in tk["nhom"].items():
    print(ten)
    display(pd.DataFrame([{"nhóm": g, "n": v["n"], "video": v["video"], **{x: D._o(v[c].get(x, 0), v["n"], MO) for c, x in COT}}
                          for g, v in sorted(gs.items(), key=lambda u: -u[1]["n"])]))

## 6. A chưa chắc ở chiều nào

In [ ]:
print(f"{tk['co_chua_chac']}/{tk['n']} bình luận có ít nhất một dấu ?")
pd.DataFrame([{"chiều": ten, "số bình luận có ?": tk["chua_chac"].get(c, 0)} for c, ten in CHIEU.values()])

## 7. Đối chiếu với mã tự do của A và B

Nhãn bên kia suy bằng luật từ khóa, nên lệch có thể do luật. Chỉ là chỉ báo, không thay cho lô K mù.

In [ ]:
for ten, p in (("mã tự do cũ của A", D.PHIEU_A), ("mã tự do của B", D.PHIEU_B)):
    r = D.doi_chieu(goc, doc_csv(p))
    print(f"{ten}: n = {r['n']}; chiều 1 đồng ý thô {100 * r['dong_y_1']:.1f}%, kappa {r['kappa_1']:.2f}")
    display(pd.DataFrame(r["nhan"]).round(2))